In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, ResNet50V2, DenseNet121
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
import warnings

In [ ]:
dataset_path = "plantvillage_dataset/color"
IMAGE_SIZE = 128
batch_size = 4
n_classes = 38
epochs = 10
warnings.filterwarnings("ignore", category=UserWarning, module="tensorflow")

In [ ]:
# Training data augmentation
train_datagen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=30,
            width_shift_range=0.2,
            height_shift_range=0.2,
            horizontal_flip=True,
            vertical_flip=True,
            zoom_range=0.2,
            shear_range=0.2,
            brightness_range=[0.8, 1.2],
            fill_mode='nearest',
            validation_split=0.2  
        )
        

In [ ]:
test_datagen = ImageDataGenerator(
            rescale=1./255,
            validation_split=0.2 
        )

In [ ]:
# Training generator
train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=batch_size,
    class_mode='categorical',
    color_mode='rgb',  
    subset='training',
    shuffle=True
)

In [ ]:
# Validation generator
validation_generator = test_datagen.flow_from_directory(
    dataset_path,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=batch_size,
    class_mode='categorical',
    color_mode='rgb',
    subset='validation',
    shuffle=False
)

In [ ]:
# Define the model_type variable
model_type = 'efficientnet' 

if model_type == 'efficientnet':
    from tensorflow.keras.applications import MobileNetV2
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
    )
elif model_type == 'resnet':
    from tensorflow.keras.applications import ResNet50
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
    )
elif model_type == 'densenet':
    from tensorflow.keras.applications import DenseNet121
    base_model = DenseNet121(
        weights='imagenet',
        include_top=False,
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
    )
else:
    raise ValueError(f"Unknown model type: {model_type}")

base_model.trainable = False
        

In [ ]:
# Build the CNN model
CHANNEL=3
model = models.Sequential([
    layers.Input(shape=(IMAGE_SIZE,IMAGE_SIZE,CHANNEL)),
    # First Convolutional Block
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    # Second Convolutional Block
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    # Third Convolutional Block
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    # Fourth Convolutional Block
    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    # Flatten and Dense Layers
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.BatchNormalization(),
    layers.Dense(n_classes, activation='softmax')
])
model.summary()

In [ ]:
# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
callbacks_list = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7
    ),
    tf.keras.callbacks.ModelCheckpoint(
        f'best_{model_type}_model.keras',
        monitor='val_accuracy',
        save_best_only=True
    )
]

In [ ]:
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator,
    callbacks=callbacks_list,
    verbose=1
    )

In [ ]:
model.layers[0].trainable = True

In [ ]:
fine_tune_at = int(len(model.layers[0].layers) * 0.8)


# Unfreeze the last few layers of the base model

for layer in model.layers[0].layers[:fine_tune_at]:
    layer.trainable = False
        
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
    )

fine_tune_epochs = 5

history_fine = model.fit(
    train_generator,
    epochs=fine_tune_epochs,
    validation_data=validation_generator,
    callbacks=callbacks_list,
    verbose=1
    )

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
import gc
import warnings
warnings.filterwarnings('ignore')

# Memory optimization settings
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
tf.config.optimizer.set_jit(True)

# Configuration - Optimized for low RAM
dataset_path = "plantvillage_dataset/color"
img_size = (128, 128)  # Reduced from 224x224 to save memory
epochs = 30
batch_size = 16  # Reduced batch size for low RAM
validation_split = 0.2
test_split = 0.2

# Disease solutions database
disease_solutions = {
    "Apple___Apple_scab": {
        "symptoms": "Dark, scabby lesions on leaves and fruit",
        "causes": "Fungal infection (Venturia inaequalis)",
        "solutions": ["Apply fungicides containing myclobutanil or captan", "Remove fallen leaves and debris"],
        "prevention": "Regular pruning, proper spacing, and sanitation"
    },
    "Apple___Black_rot": {
        "symptoms": "Brown to black circular lesions on fruit and leaves",
        "causes": "Fungal infection (Botryosphaeria obtusa)",
        "solutions": ["Remove infected plant material immediately", "Apply copper-based fungicides"],
        "prevention": "Proper sanitation and regular monitoring"
    },
    "Tomato___Early_blight": {
        "symptoms": "Concentric rings on leaves, stem lesions",
        "causes": "Fungal infection (Alternaria solani)",
        "solutions": ["Apply chlorothalonil or mancozeb fungicides", "Remove infected leaves"],
        "prevention": "Crop rotation and proper plant spacing"
    },
    "Tomato___healthy": {
        "symptoms": "No visible disease symptoms",
        "causes": "Healthy plant",
        "solutions": ["Continue current care practices", "Maintain regular watering"],
        "prevention": "Continue preventive care and monitoring"
    }
}

print("Plant Disease Recognition System - Low RAM Mode")


# Verify dataset path exists
if not os.path.exists(dataset_path):
    print(f"ERROR: Dataset path '{dataset_path}' does not exist!")
    print("Please check your dataset path and try again.")
    exit(1)

# Check if dataset has subdirectories
subdirs = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
if len(subdirs) == 0:
    print(f"ERROR: No subdirectories found in '{dataset_path}'!")
    print("Dataset should have subdirectories for each class.")
    exit(1)

print(f"\nFound {len(subdirs)} class directories in dataset")

# Clean dataset - remove corrupted images
print("\nChecking dataset for corrupted images...")
corrupted_count = 0
for subdir in subdirs:
    class_path = os.path.join(dataset_path, subdir)
    image_files = [f for f in os.listdir(class_path) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    for img_file in image_files:
        img_path = os.path.join(class_path, img_file)
        try:
            img = cv2.imread(img_path)
            if img is None:
                print(f"Removing corrupted image: {img_path}")
                os.remove(img_path)
                corrupted_count += 1
        except Exception as e:
            print(f"Error checking {img_path}: {e}")
            try:
                os.remove(img_path)
                corrupted_count += 1
            except:
                pass

if corrupted_count > 0:
    print(f"Removed {corrupted_count} corrupted images")
else:
    print("All images are valid!")

# Use ImageDataGenerator to load data directly from directory (memory efficient)
print("\nSetting up data generators (memory-efficient mode)...")

# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.15,
    validation_split=validation_split
)

# Only rescaling for validation/test
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=validation_split
)

# Create training generator with error handling
try:
    train_generator = train_datagen.flow_from_directory(
        dataset_path,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='categorical',
        subset='training',
        shuffle=True,
        interpolation='bilinear'
    )
except Exception as e:
    print(f"ERROR creating training generator: {e}")
    print("\nPlease ensure:")
    print("1. Dataset path is correct")
    print("2. Images are organized in subdirectories by class")
    print("3. Image files are valid (jpg, jpeg, png)")
    exit(1)

# Create validation generator with error handling
try:
    validation_generator = val_datagen.flow_from_directory(
        dataset_path,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='categorical',
        subset='validation',
        shuffle=False,
        interpolation='bilinear'
    )
except Exception as e:
    print(f"ERROR creating validation generator: {e}")
    exit(1)

# Get class names
class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)

print(f"\nFound {num_classes} classes")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")

# Create lightweight MobileNetV2 model
print("\nCreating MobileNetV2 model (optimized for low RAM)...")

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(*img_size, 3),
    alpha=0.75  # Reduced width multiplier to save memory
)

# Freeze base model layers
base_model.trainable = False

# Build lightweight model
model = models.Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

# Compile model
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nModel created successfully!")
print(f"Total parameters: {model.count_params():,}")

# Callbacks
callbacks_list = [
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        'best_model_mobilenetv2.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# Train model with batch processing
print("\nTraining model (batch processing mode)...")
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    epochs=epochs,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size,
    callbacks=callbacks_list,
    verbose=1,
    max_queue_size=5,  # Limit queue size to save memory
    workers=1,  # Reduce workers to save memory
    use_multiprocessing=False
)

# Clear memory
gc.collect()
tf.keras.backend.clear_session()

print("\nInitial training completed!")

# Fine-tuning with batch processing
print("\nStarting fine-tuning...")
base_model.trainable = True

# Fine-tune only last 20 layers
fine_tune_at = len(base_model.layers) - 20
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Reset generators
train_generator.reset()
validation_generator.reset()

# Continue training
history_fine = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    epochs=15,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size,
    callbacks=callbacks_list,
    verbose=1,
    max_queue_size=5,
    workers=1,
    use_multiprocessing=False
)

# Evaluate model in batches
print("\nEvaluating model...")
validation_generator.reset()
results = model.evaluate(validation_generator, verbose=1)
print(f"\nValidation Loss: {results[0]:.4f}")
print(f"Validation Accuracy: {results[1]:.4f}")

# Generate predictions in batches
print("\nGenerating predictions...")
validation_generator.reset()
y_pred = model.predict(
    validation_generator,
    steps=validation_generator.samples // batch_size,
    verbose=1
)
y_pred_classes = np.argmax(y_pred, axis=1)

# Get true labels
y_true = validation_generator.classes[:len(y_pred_classes)]

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred_classes, 
                          target_names=class_names, 
                          zero_division=0))

# Generate and save confusion matrix
print("\nGenerating confusion matrix...")
cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, cbar=True)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.close()
print("Confusion matrix saved as 'confusion_matrix.png'")

# Clear memory
del y_pred, cm
gc.collect()

# Plot training history
print("\nGenerating training history plots...")
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=100, bbox_inches='tight')
plt.close()
print("Training history plots saved as 'training_history.png'")

# Save final model
model.save('plant_disease_mobilenetv2_final.h5')
print("\nModel saved as 'plant_disease_mobilenetv2_final.h5'")

# Save class names for later use
with open('class_names.txt', 'w') as f:
    for class_name in class_names:
        f.write(f"{class_name}\n")
print("Class names saved as 'class_names.txt'")

# Clear memory
gc.collect()
tf.keras.backend.clear_session()

print("\n" + "=" * 50)
print("Training completed successfully!")
print(f"Final Validation Accuracy: {results[1]:.4f}")
print("=" * 50)

print("\n--- Memory-Efficient Prediction Example ---")
print("""
# To make predictions on a new image (memory-efficient):

import cv2
import numpy as np
import tensorflow as tf

# Load model
model = tf.keras.models.load_model('plant_disease_mobilenetv2_final.h5')

# Load class names
with open('class_names.txt', 'r') as f:
    class_names = [line.strip() for line in f]

# Load and preprocess image
test_image_path = 'path_to_your_test_image.jpg'
img = cv2.imread(test_image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (128, 128))
img = img.astype(np.float32) / 255.0
img = np.expand_dims(img, axis=0)

# Make prediction
prediction = model.predict(img, verbose=0)
class_idx = np.argmax(prediction[0])
confidence = prediction[0][class_idx]
predicted_class = class_names[class_idx]

print(f"Predicted Disease: {predicted_class}")
print(f"Confidence: {confidence:.4f}")
""")